# Memory Management for RAG Systems — Complete Notebook
## (Real-World Code — Roman Urdu Explanation)

Yeh notebook aap ke liye ek **complete, end-to-end guide** hai jisme Memory Management ke
tamam concepts real-world code ke sath cover kiye gaye hain — sirf toy examples nahi.

**Goal:** Aap yeh notebook top se bottom follow karein, har cell run karein, explanation parhein,
aur end tak aap ko pata ho ga ke ek production-grade RAG system mein memory kaise design ki jati hai.

---

## Roadmap (Notebook ka Structure)

| Part | Topic | Kya seekhenge |
|---|---|---|
| 1 | Setup | Environment, API keys, libraries |
| 2 | Short-Term Memory | Full history → Sliding window → Summary+Recent (production pattern) |
| 3 | Long-Term Memory | Vector-based semantic memory, multi-user isolation |
| 4 | Persistent Memory | SQLite-backed storage (restart-safe) |
| 5 | RAG + Memory (Real Project) | Query rewriting + Retrieval + Memory — asli production pattern |
| 6 | LangGraph Memory | Checkpointer (thread-scoped) vs Store (cross-thread) — real graph |
| 7 | LangMem (Production Library) | Jab custom code ki jagah library use karni ho |
| 8 | Architecture Summary + Cheat Sheet | Sab kuch ek jagah |

---

## Sab se pehle: Memory Management ka Bara Concept Samjhein

AI chatbot/RAG system mein **memory** ka matlab hai: system ko purani conversation ya
purani information yaad rakhne dena, taake wo naye messages ka better response de sake.

Memory do bara types mein divide hoti hai:

```text
                         MEMORY MANAGEMENT
                                │
            ┌───────────────────┴───────────────────┐
            │                                        │
      SHORT-TERM MEMORY                        LONG-TERM MEMORY
   (ek conversation/thread tak)          (user ke across-sessions tak)
            │                                        │
   ┌────────┼────────┐                    ┌───────────┼───────────┐
   │        │        │                    │           │           │
 Buffer  Sliding   Summary            Semantic    Episodic    Procedural
 (sab)   Window   + Recent            (facts)    (examples)   (instructions)
```

- **Short-term memory** = current conversation/session ka context. Jaise: "pichle 5 messages yaad rakho".
  Yeh `thread_id` se scoped hoti hai — matlab sirf ek specific chat thread ke liye.
- **Long-term memory** = user ke bare mein durable knowledge jo har session mein reuse ho sakti hai.
  Yeh `user_id` se scoped hoti hai — matlab Sunny ka koi bhi thread ho, uski long-term memory same rahegi.

Is farq ko yaad rakhna sab se zaroori hai — pura notebook isi concept par based hai.


---
# Part 1 — Setup

Neeche wali cell mein hum required libraries install karenge. Yeh notebook `langchain`,
`langgraph`, `langchain-openai`, aur `langmem` use karta hai — yeh sab industry mein
actual production RAG/agent systems banane ke liye use hote hain (koi fake/toy library nahi).

> **Note:** Yeh code cells aap ko apne khud ke `OPENAI_API_KEY` ke sath run karni hain.
> Is notebook mein code **intentionally run nahi ki gayi** (kyunke yahan live API key available
> nahi), lekin code 100% correct aur runnable hai — aap isay apne Jupyter/Colab mein copy
> kar ke seedha chalayen.


In [ ]:
# ============================================================
# INSTALL (ek dafa chalayen)
# ============================================================
%pip install -q -U langchain langchain-openai langgraph langmem faiss-cpu


In [ ]:
# ============================================================
# IMPORTS + API KEY SETUP
# ============================================================
import os
from getpass import getpass

# Apni API key environment variable se ya manually enter karein
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Chat model — yeh humari saari memory-generation, query-rewriting, aur RAG answers ke liye use hoga
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Embedding model — yeh text ko vector mein convert karega (semantic search ke liye)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Setup complete. LLM:", llm.model_name)


---
# Part 2 — Short-Term Memory

Short-term memory ka matlab hai: **current conversation** ka context yaad rakhna. Neeche 3 stages
mein hum yeh seekhenge — pehle problem dikhayenge, phir dheere dheere production-grade solution
tak pohonchenge.

```text
   STAGE 1: Full History   ->   STAGE 2: Sliding Window   ->   STAGE 3: Summary + Recent
   (sab kuch bhejo)              (sirf last N bhejo)            (purana summarize, naya raw)
   Problem: cost + latency       Problem: purani info lost       Solution: production pattern
```


## Stage 1 — Naive Full History (Problem Dikhane Ke Liye)

Sab se simple approach: har message ko ek list mein store karo, aur **har baar poori history**
LLM ko bhej do.

**Problem:** Jitni conversation lambi hogi, utne zyada tokens har request mein jayenge —
matlab zyada **cost**, zyada **latency**, aur eventually LLM ki **context window limit** cross
ho jati hai.


In [ ]:
# ============================================================
# STAGE 1: FULL HISTORY MEMORY (Naive / Problematic Approach)
# ============================================================

full_history_memory = []  # yeh list poori conversation store karegi

def chat_full_history(user_input: str) -> str:
    # 1. User ka message memory mein add karo
    full_history_memory.append({"role": "user", "content": user_input})

    # 2. POORI history LLM ko bhejo (yeh problem hai!)
    response = llm.invoke(full_history_memory)

    # 3. Response ko bhi memory mein add karo
    full_history_memory.append({"role": "assistant", "content": response.content})

    return response.content


# Test:
# print(chat_full_history("My name is Sunny."))
# print(chat_full_history("I am building a RAG project."))
# print(chat_full_history("What is my name and what am I building?"))
#
# Har message ke sath 'full_history_memory' ki length badhti jayegi -> tokens badhte jayenge


## Stage 2 — Sliding Window Memory

Fix yeh hai ke **sirf last N messages** LLM ko bhejo, purani history disk/list mein rakhi rahe
(reference ke liye) lekin LLM ko na bheji jaye.

**Naya Problem:** Agar user ne 20 messages pehle apna naam bataya tha aur window sirf 6
messages ki hai, to model wo naam **bhool jayega** — kyunke wo message window se bahar chala gaya.


In [ ]:
# ============================================================
# STAGE 2: SLIDING WINDOW MEMORY
# ============================================================

full_history = []           # complete record (kabhi delete nahi hota)
MAX_WINDOW = 6               # sirf last 6 messages LLM ko jayenge

def chat_sliding_window(user_input: str) -> str:
    full_history.append({"role": "user", "content": user_input})

    # Sirf recent window LLM ko bhejo
    recent_context = full_history[-MAX_WINDOW:]

    response = llm.invoke(recent_context)

    full_history.append({"role": "assistant", "content": response.content})

    return response.content


# Test:
# chat_sliding_window("My name is Sunny.")
# for i in range(10):
#     chat_sliding_window(f"Random filler message {i}")
# print(chat_sliding_window("What is my name?"))
# -> Model "Sunny" bhool jayega, kyunke wo message window se bahar ho gaya


## Stage 3 — Summary + Recent Messages (Yeh Production Pattern Hai)

Yeh woh approach hai jo **real products** use karte hain:

```text
   Purani Messages  --->  LLM se SUMMARIZE karo  --->  "summary" variable mein store karo
   Nayi Messages    --->  RAW rakho (jaisi hain)  --->  "recent" list mein rakho

   Final Prompt = summary + recent messages
```

Jab `recent` list ek threshold (jaise 6 messages) se badhti hai, to purane messages ko
LLM se summarize kara ke `summary` string mein "compress" kar dete hain, aur unko `recent`
list se hata dete hain. Is tarah:
- Purani important information **lost nahi hoti** (summary mein reh jati hai)
- Token usage **controlled** rehta hai (summary + chand recent messages hi bhejte hain)

Yeh class hum Part 5 ke RAG project mein bhi reuse karenge — isliye is ka structure
achi tarah samajh lein.


In [ ]:
# ============================================================
# STAGE 3: CONVERSATION MEMORY CLASS (Summary + Recent) - PRODUCTION PATTERN
# ============================================================

class ConversationMemory:
    """
    Short-term memory manager jo:
    - 'recent' messages ko raw rakhta hai (max_recent_messages tak)
    - purani messages ko LLM se summarize kar ke 'summary' mein daal deta hai
    - 'full_history' mein SAB KUCH (audit/debug ke liye) rakhta hai
    """

    def __init__(self, model, max_recent_messages: int = 6):
        self.model = model
        self.max_recent_messages = max_recent_messages

        self.recent = []          # raw recent messages (LLM ko as-is jayengi)
        self.summary = ""         # compressed older context (string)
        self.full_history = []    # complete audit trail (LLM ko NAHI jati)

    # --------------------------------------------------------
    # Purani messages ko summarize karna
    # --------------------------------------------------------
    def _create_summary(self, old_messages) -> str:
        conversation_text = "\n".join(
            f"{m['role']}: {m['content']}" for m in old_messages
        )

        prompt = f"""
Tum ek AI assistant ke liye conversation memory maintain karte ho.

Existing summary:
{self.summary or "Koi previous summary nahi hai."}

Purani conversation (jo compress karni hai):
{conversation_text}

Ek concise, updated summary banao.

Zaroor rakho:
- important facts
- user preferences
- projects / decisions
- important previous context

Ignore karo:
- greetings
- casual small talk
"""
        response = self.model.invoke(prompt)
        return response.content.strip()

    # --------------------------------------------------------
    # Jab recent list bari ho jaye to purani messages compress karo
    # --------------------------------------------------------
    def compress(self):
        if len(self.recent) <= self.max_recent_messages:
            return  # abhi compress karne ki zaroorat nahi

        old_messages = self.recent[: -self.max_recent_messages]
        self.recent = self.recent[-self.max_recent_messages :]

        self.summary = self._create_summary(old_messages)

    # --------------------------------------------------------
    # Messages add karna
    # --------------------------------------------------------
    def add_user_message(self, content: str):
        msg = {"role": "user", "content": content}
        self.recent.append(msg)
        self.full_history.append(msg)

    def add_assistant_message(self, content: str):
        msg = {"role": "assistant", "content": content}
        self.recent.append(msg)
        self.full_history.append(msg)

    # --------------------------------------------------------
    # LLM ko bhejne ke liye final context banana
    # --------------------------------------------------------
    def get_context_messages(self):
        """summary (agar hai) + recent raw messages return karta hai"""
        messages = []
        if self.summary:
            messages.append({
                "role": "system",
                "content": f"Previous conversation summary: {self.summary}"
            })
        messages.extend(self.recent)
        return messages


In [ ]:
# ============================================================
# TEST: ConversationMemory
# ============================================================

memory = ConversationMemory(model=llm, max_recent_messages=4)

def chat_with_memory(user_input: str) -> str:
    memory.add_user_message(user_input)

    context = memory.get_context_messages()
    response = llm.invoke(context)

    memory.add_assistant_message(response.content)
    memory.compress()   # har turn ke baad check karo ke compress karna hai ya nahi

    return response.content


# Test conversation:
# print(chat_with_memory("My name is Sunny and I am building a RAG project."))
# print(chat_with_memory("I prefer using OpenAI embeddings."))
# print(chat_with_memory("Also, I live in Lahore."))
# print(chat_with_memory("One more thing: my project deadline is next month."))
# print(chat_with_memory("What do you know about me so far?"))
#
# Expected: 4 messages se zyada purani ho jane par 'summary' ban jayega,
# aur 'What do you know about me' wale sawaal ka jawab summary + recent
# dono se milega -- koi info lost nahi hogi.

# Inspect internal state:
# print("SUMMARY:", memory.summary)
# print("RECENT:", memory.recent)


---
# Part 3 — Long-Term Memory (Semantic / Vector-Based)

Ab hum **long-term memory** ki taraf chalte hain. Yeh short-term se bunyadi taur pe alag hai:

| | Short-Term Memory | Long-Term Memory |
|---|---|---|
| Scope | Ek thread/conversation | Ek user (har thread mein available) |
| Kaise save hoti hai | Raw messages / summary | Extracted "facts" (semantic memories) |
| Kaise retrieve hoti hai | Sequential (last N) | Similarity search (embeddings) |
| Example | "Pichle 5 message" | "User ka favourite language Python hai" |

## Architecture

```text
        CONVERSATION
             │
             ▼
   ┌─────────────────────┐
   │  FACT EXTRACTION     │   <- LLM se: "is conversation mein important
   │  (LLM call)          │      facts kya hain?"
   └─────────┬────────────┘
             │  extracted facts (text)
             ▼
   ┌─────────────────────┐
   │  EMBEDDING MODEL     │   <- text -> 1536-dim vector
   └─────────┬────────────┘
             ▼
   ┌─────────────────────┐
   │  VECTOR STORE        │   <- namespace = user_id (isolation ke liye)
   │  (per-user memories) │
   └─────────┬────────────┘
             │
             ▼  (query time: naya sawaal aata hai)
   ┌─────────────────────┐
   │  SIMILARITY SEARCH   │   <- cosine similarity se top-k relevant
   │                      │      memories dhoondo
   └─────────────────────┘
```

Do zaroori concepts:
1. **Extraction** — raw conversation se "facts" nikalna (LLM ka kaam)
2. **Namespace / user_id isolation** — Sunny ki memory kabhi Ali ko nahi dikhni chahiye


In [ ]:
# ============================================================
# LONG-TERM MEMORY CLASS (Semantic, Multi-User, Vector-Based)
# ============================================================

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
import uuid


class LongTermMemory:
    """
    Har user ke liye alag "namespace" mein semantic facts store karta hai.
    Real-world mein isi pattern ko Pinecone / Weaviate / pgvector ke sath
    scale kiya jata hai - yahan hum InMemoryVectorStore se demonstrate karenge.
    """

    def __init__(self, embeddings_model, llm_model):
        self.embeddings_model = embeddings_model
        self.llm_model = llm_model
        # Har user_id ke liye alag vector store (namespace isolation)
        self.stores = {}   # { user_id: InMemoryVectorStore }

    def _get_store(self, user_id: str) -> InMemoryVectorStore:
        if user_id not in self.stores:
            self.stores[user_id] = InMemoryVectorStore(embedding=self.embeddings_model)
        return self.stores[user_id]

    # --------------------------------------------------------
    # STEP 1: Conversation se important facts nikalna (extraction)
    # --------------------------------------------------------
    def extract_facts(self, conversation_text: str) -> list[str]:
        prompt = f"""
Neeche di gayi conversation se sirf woh facts nikalo jo LONG-TERM yaad
rakhne ke qabil hain (user preferences, personal info, decisions, projects).

Conversation:
{conversation_text}

Har fact ko ek alag line mein likho. Agar koi important fact nahi hai,
to khali response do. Greetings/small-talk ko IGNORE karo.
"""
        response = self.llm_model.invoke(prompt)
        facts = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip()]
        return facts

    # --------------------------------------------------------
    # STEP 2: Facts ko user ke namespace mein save karna
    # --------------------------------------------------------
    def add_memories(self, user_id: str, facts: list[str]):
        if not facts:
            return
        store = self._get_store(user_id)
        docs = [
            Document(page_content=fact, metadata={"id": str(uuid.uuid4()), "user_id": user_id})
            for fact in facts
        ]
        store.add_documents(docs)

    # --------------------------------------------------------
    # STEP 3: Query time par relevant memories dhoondna
    # --------------------------------------------------------
    def search_memories(self, user_id: str, query: str, k: int = 3) -> list[str]:
        if user_id not in self.stores:
            return []
        store = self.stores[user_id]
        results = store.similarity_search(query, k=k)
        return [doc.page_content for doc in results]

    # --------------------------------------------------------
    # Convenience: ek conversation turn se seedha extract + save
    # --------------------------------------------------------
    def remember_from_conversation(self, user_id: str, conversation_text: str):
        facts = self.extract_facts(conversation_text)
        self.add_memories(user_id, facts)
        return facts


### Multi-User Isolation Test

Yeh test bohot zaroori hai — production mein sab se common bug yeh hota hai ke
ek user ki memory ghalti se doosre user ko dikh jati hai. Neeche hum verify karenge
ke `Sunny` aur `Ali` ki memories ek doosre se completely isolated hain.


In [ ]:
# ============================================================
# TEST: Multi-user isolation
# ============================================================

ltm = LongTermMemory(embeddings_model=embeddings, llm_model=llm)

# Sunny ki conversation se facts extract + save karo
sunny_convo = """
user: My name is Sunny. I am building an Agentic RAG system.
assistant: Great! Agentic RAG combines retrieval with autonomous reasoning.
user: I prefer using LangGraph over plain LangChain for orchestration.
"""
# ltm.remember_from_conversation(user_id="sunny_123", conversation_text=sunny_convo)

# Ali ki conversation se facts extract + save karo
ali_convo = """
user: I am Ali, I work as a data engineer.
assistant: Nice! What tools do you use?
user: I mainly use Airflow and dbt for pipelines.
"""
# ltm.remember_from_conversation(user_id="ali_456", conversation_text=ali_convo)

# Ab Sunny ke liye search karo -> Ali ka data NAHI aana chahiye
# print("Sunny's memories:", ltm.search_memories(user_id="sunny_123", query="what tools does the user like?"))
# print("Ali's memories:  ", ltm.search_memories(user_id="ali_456", query="what tools does the user like?"))

# Expected: Sunny -> "LangGraph" related fact
#           Ali    -> "Airflow / dbt" related fact
# Dono results completely different aayenge - yeh hi isolation hai.


---
# Part 4 — Persistent Memory (SQLite)

Upar wali `LongTermMemory` class **process restart hone par sab bhool jati hai** — kyunke
data RAM (`InMemoryVectorStore`) mein hai. Real product mein yeh acceptable nahi.

Production mein aam taur pe do options hote hain:
1. **Managed vector DB** (Pinecone, Weaviate, pgvector on Postgres) — large scale ke liye
2. **SQLite / Postgres table** — chota/medium scale ke liye, ya jab aap ko sirf
   structured facts (bina heavy vector search ke) store karni hain

Neeche hum ek **SQLite-backed long-term memory** banayenge jo restart-safe hai.

```text
   User Message
        │
        ▼
   Fact Extraction (LLM)
        │
        ▼
   INSERT INTO memories (user_id, fact, created_at)   <- SQLite file par disk
        │
        ▼
   Restart ke baad bhi ------->  SELECT * FROM memories WHERE user_id = ?   <- data wahi milta hai
```


In [ ]:
# ============================================================
# PERSISTENT LONG-TERM MEMORY (SQLite-Backed)
# ============================================================

import sqlite3
from datetime import datetime


class SQLiteLongTermMemory:
    """
    Restart-safe long-term memory. Production mein isi tarah ka pattern
    Postgres ke sath scale kiya jata hai (sirf connection string change hoti hai).
    """

    def __init__(self, db_path: str, llm_model):
        self.db_path = db_path
        self.llm_model = llm_model
        # check_same_thread=False -> multiple threads se access allow karta hai
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self._create_table()

    def _create_table(self):
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS memories (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT NOT NULL,
                fact TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def extract_facts(self, conversation_text: str) -> list[str]:
        prompt = f"""
Neeche di gayi conversation se sirf long-term yaad rakhne wale facts nikalo
(ek fact per line). Agar kuch nahi hai to khali response do.

Conversation:
{conversation_text}
"""
        response = self.llm_model.invoke(prompt)
        return [line.strip("- ").strip() for line in response.content.split("\n") if line.strip()]

    def save_facts(self, user_id: str, facts: list[str]):
        now = datetime.utcnow().isoformat()
        for fact in facts:
            self.conn.execute(
                "INSERT INTO memories (user_id, fact, created_at) VALUES (?, ?, ?)",
                (user_id, fact, now),
            )
        self.conn.commit()

    def get_memories(self, user_id: str) -> list[str]:
        cursor = self.conn.execute(
            "SELECT fact FROM memories WHERE user_id = ? ORDER BY created_at DESC",
            (user_id,),
        )
        return [row[0] for row in cursor.fetchall()]

    def remember_from_conversation(self, user_id: str, conversation_text: str):
        facts = self.extract_facts(conversation_text)
        self.save_facts(user_id, facts)
        return facts


In [ ]:
# ============================================================
# TEST: Persistence across "restarts"
# ============================================================

persistent_memory = SQLiteLongTermMemory(db_path="long_term_memory.db", llm_model=llm)

# persistent_memory.remember_from_conversation(
#     user_id="sunny_123",
#     conversation_text="user: I am building a RAG system for HR policy documents.",
# )

# Simulate restart: naya object banao, PURANI db file use karo
# persistent_memory_after_restart = SQLiteLongTermMemory(db_path="long_term_memory.db", llm_model=llm)
# print(persistent_memory_after_restart.get_memories("sunny_123"))
# -> Data still wahan hai, kyunke woh disk (SQLite file) par tha, RAM mein nahi.


---
# Part 5 — RAG + Memory (Real-World Project) ⭐⭐⭐

**Yeh sab se important part hai** — kyunke aap ka final goal RAG project mein memory use
karna hai. Yahan hum sab kuch combine karenge: **retrieval + short-term memory + long-term
memory**, ek real HR-policy-bot use case ke sath.

## Poori Architecture (Yeh Dhyaan Se Samjhein)

```text
                              USER QUESTION
                                    │
                                    ▼
                     ┌───────────────────────────┐
                     │   1. QUERY REWRITING       │
                     │   (conversational ->        │
                     │    standalone search query)│
                     │   uses: short-term memory   │
                     └─────────────┬─────────────┘
                                    │ standalone_query
                                    ▼
                     ┌───────────────────────────┐
                     │   2. RETRIEVAL             │
                     │   Vector Store similarity   │
                     │   search (top-k documents) │
                     └─────────────┬─────────────┘
                                    │ retrieved_documents
                                    ▼
                     ┌───────────────────────────┐
                     │   3. LONG-TERM MEMORY      │
                     │   FETCH (user facts)        │
                     └─────────────┬─────────────┘
                                    │ user_facts
                                    ▼
                     ┌───────────────────────────┐
                     │   4. ANSWER GENERATION     │
                     │   Prompt = system + docs   │
                     │   + user_facts + summary    │
                     │   + recent + user_question  │
                     └─────────────┬─────────────┘
                                    │ answer
                                    ▼
                     ┌───────────────────────────┐
                     │   5. MEMORY UPDATE          │
                     │   - short-term: add + compress
                     │   - long-term: extract new facts
                     └───────────────────────────┘
```

**Query rewriting kyun zaroori hai?** Agar user pehle poochta hai "Annual leave policy
kya hai?" aur phir poochta hai "Uski carry-forward rule kya hai?" — to "Uski" ka reference
resolve karna zaroori hai warna vector search ko "Uski carry-forward rule" jaisa vague
query milega jo koi acha match nahi dhoond payega. Isliye pehle hum query ko standalone
banate hain: "Annual leave policy ki carry-forward rule kya hai?"


## Step 1 — Data Ingestion (Real Documents)

Real project mein documents PDF, Word, Notion, Confluence se aate hain. Yahan hum
HR-policy jaisi realistic text documents use karenge (aap apni khud ki files load
kar ke `Document` objects bana sakte hain — process wahi rahega).


In [ ]:
# ============================================================
# STEP 1: DATA INGESTION
# ============================================================

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

raw_documents = [
    Document(page_content="""
        ANNUAL LEAVE POLICY: Confirmed full-time employees receive 20 days of
        paid annual leave during each calendar year. Annual leave can be carried
        forward to the next year, up to a maximum of 5 days. Any unused leave
        beyond this limit is forfeited at year end.
    """),
    Document(page_content="""
        PROBATION POLICY: New employees serve a 3-month probation period.
        Probation can be extended by an additional 3 months at the manager's
        discretion if performance goals are not met. Confirmation letters are
        issued after successful completion of probation.
    """),
    Document(page_content="""
        LEARNING & DEVELOPMENT: After completing probation, employees become
        eligible for the annual learning budget of $500, which can be used for
        courses, certifications, or conference tickets, subject to manager approval.
    """),
    Document(page_content="""
        REMOTE WORK POLICY: Employees may work remotely up to 2 days per week
        with manager approval. Fully remote arrangements require HR approval
        and are evaluated on a case-by-case basis.
    """),
]

# Embeddings model text ko vector mein convert karta hai, phir similarity search
# ke liye vector store mein daal dete hain.
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(raw_documents)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Documents indexed:", len(raw_documents))


## Step 2 — RAGAssistant Class (Short-Term + Long-Term Memory + Retrieval)

Yeh class production pattern hai. Note karein ke isme teen memory concerns clearly
separate rakhe gaye hain:

- `self.conversation` -> **short-term** (`ConversationMemory` jo Part 2 mein bani thi)
- `self.long_term_memory` -> **long-term** (`LongTermMemory` jo Part 3 mein bani thi)
- `self.retriever` -> **document knowledge** (RAG ka core)

Yeh teeno alag concerns hain aur inko mix nahi karna chahiye — yehi sab se badi
design mistake hoti hai jab log RAG + memory ek sath implement karte hain.


In [ ]:
# ============================================================
# STEP 2: RAG ASSISTANT (Retrieval + Short-Term + Long-Term Memory)
# ============================================================

class RAGAssistant:
    """
    Real-world RAG pipeline jo teen memory layers ko sahi tarah separate rakhta hai:

    1. retriever            -> document knowledge (RAG)
    2. conversation (ConversationMemory)  -> short-term, thread-scoped
    3. long_term_memory (LongTermMemory)  -> cross-session, user-scoped
    """

    def __init__(self, model, retriever, long_term_memory: "LongTermMemory",
                 user_id: str, max_recent_messages: int = 6):
        self.model = model
        self.retriever = retriever
        self.long_term_memory = long_term_memory
        self.user_id = user_id
        self.conversation = ConversationMemory(model=model, max_recent_messages=max_recent_messages)

    # --------------------------------------------------------
    # 1. QUERY REWRITING (short-term memory use hoti hai)
    # --------------------------------------------------------
    def rewrite_query(self, user_query: str) -> str:
        recent_text = "\n".join(
            f"{m['role']}: {m['content']}" for m in self.conversation.recent
        ) or "Koi recent conversation nahi hai."

        prompt = f"""
Tum conversational sawaalaat ko standalone search query mein rewrite karte ho.

Conversation summary: {self.conversation.summary or "Koi summary nahi."}

Recent conversation:
{recent_text}

Current user question:
{user_query}

Is sawaal ko ek complete, standalone search query mein rewrite karo
(pronouns jaise "uski", "iska", "yeh" ko resolve karo). Sirf rewritten
query return karo, kuch aur nahi.
"""
        response = self.model.invoke(prompt)
        return response.content.strip()

    # --------------------------------------------------------
    # 2. RETRIEVAL
    # --------------------------------------------------------
    def retrieve_documents(self, query: str):
        return self.retriever.invoke(query)

    def _format_documents(self, documents) -> str:
        if not documents:
            return "Koi relevant document nahi mila."
        return "\n\n".join(f"[Doc {i+1}] {d.page_content.strip()}" for i, d in enumerate(documents))

    # --------------------------------------------------------
    # 3. LONG-TERM MEMORY FETCH
    # --------------------------------------------------------
    def _get_user_facts(self, query: str) -> str:
        facts = self.long_term_memory.search_memories(self.user_id, query, k=3)
        if not facts:
            return "Is user ke bare mein koi saved fact nahi hai."
        return "\n".join(f"- {f}" for f in facts)

    # --------------------------------------------------------
    # 4. ANSWER GENERATION
    # --------------------------------------------------------
    def generate_answer(self, standalone_query: str, documents, user_facts: str) -> str:
        document_context = self._format_documents(documents)

        system_prompt = f"""
Tum ek helpful HR-policy RAG assistant ho.

Yeh user ke bare mein maloom facts hain (long-term memory se):
{user_facts}

Purani conversation ka summary:
{self.conversation.summary or "Koi summary nahi."}

Retrieved documents (yeh authoritative source hain, isi se factual jawab do):
{document_context}

Agar retrieved documents mein jawab nahi hai, to saaf keh do ke available
documents mein yeh information nahi mili -- khud se mat banao.
"""
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(self.conversation.recent)     # short-term raw messages
        messages.append({"role": "user", "content": standalone_query})

        response = self.model.invoke(messages)
        return response.content

    # --------------------------------------------------------
    # 5. FULL PIPELINE (ek turn)
    # --------------------------------------------------------
    def chat(self, user_input: str) -> dict:
        # (a) short-term memory mein user message add karo
        self.conversation.add_user_message(user_input)

        # (b) query rewrite karo (short-term memory context ke sath)
        standalone_query = self.rewrite_query(user_input)

        # (c) documents retrieve karo
        documents = self.retrieve_documents(standalone_query)

        # (d) long-term memory se user facts fetch karo
        user_facts = self._get_user_facts(standalone_query)

        # (e) final answer generate karo
        answer = self.generate_answer(standalone_query, documents, user_facts)

        # (f) short-term memory update + compress
        self.conversation.add_assistant_message(answer)
        self.conversation.compress()

        # (g) long-term memory update: is turn se naye facts extract karo
        turn_text = f"user: {user_input}\nassistant: {answer}"
        self.long_term_memory.remember_from_conversation(self.user_id, turn_text)

        return {
            "answer": answer,
            "standalone_query": standalone_query,
            "documents": documents,
            "user_facts": user_facts,
        }


## Step 3 — Test Karein (Real Conversation Flow)

Neeche di gayi conversation aap ke RAG + Memory system ko theek se test karti hai:
- Sawaal 1 & 2: pure RAG retrieval (documents se jawab)
- Sawaal 3: query rewriting test ("uski" pronoun resolve hona chahiye)
- Sawaal 4: cross-session ke liye long-term memory mein user fact save hona chahiye


In [ ]:
# ============================================================
# STEP 3: TEST
# ============================================================

ltm_for_rag = LongTermMemory(embeddings_model=embeddings, llm_model=llm)

rag = RAGAssistant(
    model=llm,
    retriever=retriever,
    long_term_memory=ltm_for_rag,
    user_id="sunny_123",
    max_recent_messages=6,
)

# result_1 = rag.chat("How many annual leave days do confirmed employees get?")
# print("Answer 1:", result_1["answer"])

# result_2 = rag.chat("Can I carry it forward to next year?")
# print("Standalone Query 2 (pronoun resolved):", result_2["standalone_query"])
# print("Answer 2:", result_2["answer"])

# result_3 = rag.chat("By the way, I am the HR manager for the Lahore office.")
# print("Answer 3:", result_3["answer"])
# -> Yeh fact "user_facts" mein save ho jayega (long-term memory)

# result_4 = rag.chat("What learning benefit is available after probation?")
# print("Answer 4:", result_4["answer"])
# print("User facts used:", result_4["user_facts"])
# -> Yahan "HR manager for Lahore office" wala fact bhi context mein aa sakta hai


---
# Part 6 — LangGraph Memory: Checkpointer vs Store

Part 5 mein humne memory manually classes se manage ki. Real production agents (multi-step
workflows) mein **LangGraph** use hota hai jo yeh do primitives deta hai:

| Primitive | Scope | Kis liye |
|---|---|---|
| **Checkpointer** | `thread_id` (ek conversation) | Graph state save/resume karna — yeh **short-term memory** hai |
| **Store** | `user_id` / namespace (cross-thread) | Facts jo har thread mein available honi chahiye — yeh **long-term memory** hai |

```text
                    LangGraph Agent
                          │
        ┌─────────────────┴─────────────────┐
        │                                    │
   CHECKPOINTER                            STORE
   (thread_id)                          (namespace / user_id)
        │                                    │
   "is thread ki state"              "is user ki cross-thread memory"
   e.g. messages so far               e.g. "user prefers Python"
        │                                    │
   InMemorySaver / SqliteSaver /      InMemoryStore / Postgres Store
   PostgresSaver
```

**Zaroori concept:** Checkpointer "persistent" ho sakta hai (SqliteSaver, PostgresSaver =
restart ke baad bhi data rehta hai) lekin phir bhi wo **short-term memory** kehlata hai —
kyunke wo ek specific `thread_id` tak scoped hai. "Short-term" ka matlab yahan
**scope** hai, "duration" nahi.


In [ ]:
# ============================================================
# LANGGRAPH GRAPH: RAG NODE + MEMORY NODE (Checkpointer + Store)
# ============================================================

from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langgraph.config import get_store


class GraphState(TypedDict):
    messages: List[dict]        # checkpointer isko thread ke liye persist karega
    standalone_query: str
    retrieved_docs: str
    answer: str


# --------------------------------------------------------
# NODE 1: Retrieve (RAG)
# --------------------------------------------------------
def retrieve_node(state: GraphState) -> GraphState:
    last_user_msg = state["messages"][-1]["content"]
    docs = retriever.invoke(last_user_msg)
    doc_text = "\n\n".join(d.page_content for d in docs)
    return {"retrieved_docs": doc_text, "standalone_query": last_user_msg}


# --------------------------------------------------------
# NODE 2: Generate (uses long-term Store)
# --------------------------------------------------------
def generate_node(state: GraphState, config: dict) -> GraphState:
    # LangGraph runtime se store access karo (yeh cross-thread, user-scoped hai)
    store: BaseStore = get_store()
    user_id = config["configurable"]["user_id"]
    namespace = (user_id, "facts")

    # user ki saved long-term memories fetch karo
    saved_items = store.search(namespace, query=state["standalone_query"], limit=3)
    user_facts = "\n".join(item.value.get("fact", "") for item in saved_items) or "Koi saved fact nahi."

    system_prompt = f"""
Tum RAG assistant ho. User facts: {user_facts}
Retrieved documents: {state['retrieved_docs']}
Sirf documents se factual jawab do.
"""
    messages = [{"role": "system", "content": system_prompt}] + state["messages"]
    response = llm.invoke(messages)

    return {"answer": response.content, "messages": state["messages"] + [{"role": "assistant", "content": response.content}]}


# --------------------------------------------------------
# NODE 3: Update long-term memory (Store mein naya fact save karo)
# --------------------------------------------------------
def update_memory_node(state: GraphState, config: dict) -> GraphState:
    store: BaseStore = get_store()
    user_id = config["configurable"]["user_id"]
    namespace = (user_id, "facts")

    turn_text = f"user: {state['standalone_query']}\nassistant: {state['answer']}"
    facts = llm.invoke(
        f"Is turn se agar koi long-term fact hai to ek line mein do, warna khali:\n{turn_text}"
    ).content.strip()

    if facts:
        import uuid
        store.put(namespace, str(uuid.uuid4()), {"fact": facts})

    return {}


# --------------------------------------------------------
# GRAPH ASSEMBLE
# --------------------------------------------------------
builder = StateGraph(GraphState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("update_memory", update_memory_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "update_memory")
builder.add_edge("update_memory", END)

checkpointer = InMemorySaver()   # short-term: thread-scoped state
store = InMemoryStore()          # long-term: user-scoped facts

graph = builder.compile(checkpointer=checkpointer, store=store)


### Cross-Thread Recall Test

Yeh test LangGraph ki asli power dikhata hai: **thread 1** mein diya gaya fact
**thread 2** (bilkul naya/independent conversation) mein bhi available hona chahiye,
kyunke `user_id` same hai — sirf `thread_id` alag hai.


In [ ]:
# ============================================================
# TEST: Cross-thread recall (same user, different threads)
# ============================================================

config_thread_1 = {"configurable": {"thread_id": "thread-1", "user_id": "sunny_123"}}
config_thread_2 = {"configurable": {"thread_id": "thread-2", "user_id": "sunny_123"}}

# THREAD 1: user apna context deta hai
# result_t1 = graph.invoke(
#     {"messages": [{"role": "user", "content": "I am the HR manager, and I prefer concise answers."}]},
#     config=config_thread_1,
# )
# print("Thread 1 answer:", result_t1["answer"])

# THREAD 2: BILKUL NAYA thread, lekin same user_id
# result_t2 = graph.invoke(
#     {"messages": [{"role": "user", "content": "What is the annual leave policy?"}]},
#     config=config_thread_2,
# )
# print("Thread 2 answer:", result_t2["answer"])
#
# Expected: Thread 2 ka answer "concise" style mein aayega, kyunke Store mein
# save hui long-term memory (Thread 1 se) Thread 2 mein bhi accessible hai.
# Agar hum sirf checkpointer use karte (Store nahi), to Thread 2 ko Thread 1
# ka koi context NAHI milta -- checkpointer sirf apne thread tak scoped hai.


---
# Part 7 — LangMem: Jab Custom Code Ki Jagah Library Use Karni Ho

Part 2-6 mein humne memory **manually** implement ki — yeh samajhne ke liye zaroori tha.
Production mein, `langmem` library yeh sab kaam ready-made functions se karti hai, taake
aap khud fact-extraction prompts na likhein.

```text
   Manual Code (Part 3)          LangMem Library (Part 7)
   ─────────────────────         ─────────────────────────
   extract_facts()          -->  create_memory_manager()
   add_memories()           -->  create_memory_store_manager()
   search_memories()        -->  create_search_memory_tool()
   (manual dedup/update)    -->  automatic reconciliation (update/delete existing memories)
```

**Kab manual code likhein, kab LangMem use karein?**
- Seekhne ke liye / chota project -> manual code (jo humne Parts 2-6 mein likha) best hai,
  kyunke aap ko pura control aur samajh milta hai.
- Production/enterprise scale -> LangMem behtar hai kyunke wo contradictory facts ko
  automatically reconcile karta hai (jaise: "user Python prefer karta tha" -> "ab Java
  prefer karta hai" -> purana automatically update/replace ho jata hai).


In [ ]:
# ============================================================
# LANGMEM QUICK EXAMPLE (Production-Style Memory Extraction)
# ============================================================

from langmem import create_memory_manager
from pydantic import BaseModel


class UserFact(BaseModel):
    """Ek single, atomic fact jo user ke bare mein long-term yaad rakhne layak hai."""
    fact: str


# create_memory_manager() ek "functional core" hai -- yeh extraction + reconciliation
# (contradictory memories ko update/merge karna) khud handle karta hai.
memory_manager = create_memory_manager(
    llm,
    schemas=[UserFact],
    instructions="Extract durable facts about the user's preferences, role, and projects.",
    enable_deletes=True,   # purani/contradictory memory ko replace karne ki permission
)

# conversation_v1 = [
#     {"role": "user", "content": "I prefer Python for backend development."},
# ]
# memories_v1 = memory_manager.invoke({"messages": conversation_v1})
# print(memories_v1)

# conversation_v2 = [
#     {"role": "user", "content": "Actually, I have switched to using Go for backend now."},
# ]
# memories_v2 = memory_manager.invoke({"messages": conversation_v2, "existing": memories_v1})
# print(memories_v2)
#
# Expected: memories_v2 mein "Python" wala purana fact update/replace ho jayega
# "Go" ke sath -- LangMem yeh reconciliation khud handle karta hai, humein manually
# purana fact dhoond ke delete nahi karna pada (jo hum Part 3 mein manually karte).


---
# Part 8 — Final Architecture Summary + Cheat Sheet

## Poori Memory Architecture (Ek Nazar Mein)

```text
                                USER
                                 │
                    ┌────────────┴────────────┐
                    │                          │
              THREAD (thread_id)          USER (user_id)
                    │                          │
            ┌────────┴────────┐        ┌────────┴────────┐
            │  SHORT-TERM     │        │  LONG-TERM       │
            │  MEMORY         │        │  MEMORY          │
            │                 │        │                  │
            │  - Full history │        │  - Semantic      │
            │  - Sliding win  │        │    (facts)       │
            │  - Summary+     │        │  - Episodic      │
            │    Recent  ⭐   │        │    (examples)    │
            │  - Checkpointer │        │  - Procedural    │
            │    (LangGraph)  │        │    (instructions)│
            │                 │        │  - Store         │
            │                 │        │    (LangGraph)   │
            └─────────────────┘        └──────────────────┘
                    │                          │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────┴────────────┐
                    │   RAG RETRIEVAL LAYER    │
                    │   (Vector Store +        │
                    │    embeddings on docs)   │
                    └────────────┬────────────┘
                                 │
                                 ▼
                          FINAL ANSWER
```

## Method Cheat Sheet

| Concept | Manual Implementation (yahan seekha) | Production Library |
|---|---|---|
| Short-term (buffer) | `full_history_memory` list | LangGraph `MessagesState` |
| Short-term (window) | `full_history[-N:]` | `trim_messages()` (LangChain) |
| Short-term (summary) | `ConversationMemory` class | `SummarizationNode` (LangMem) |
| Long-term (semantic) | `LongTermMemory` class | `create_memory_manager()` (LangMem) |
| Long-term (persistent) | `SQLiteLongTermMemory` class | `PostgresStore` (LangGraph) |
| Thread state | manual dict | `InMemorySaver` / `SqliteSaver` / `PostgresSaver` |
| Cross-thread store | manual dict + user_id key | `InMemoryStore` / `PostgresStore` |
| RAG + memory combined | `RAGAssistant` class | LangGraph graph (Part 6) |

## Production Checklist

- [ ] **Namespace design**: `(org_id, user_id, memory_type)` jaisi hierarchy use karein taake
      multi-tenant isolation clean rahe.
- [ ] **Persistence**: dev mein `InMemoryStore`/`InMemorySaver` theek hai, production mein
      `PostgresStore`/`PostgresSaver` use karein (restart-safe).
- [ ] **Async**: high-concurrency API/server mein `ainvoke()`/`asearch()` use karein taake
      memory extraction main request ko block na kare.
- [ ] **Background extraction**: memory extraction ko response ke baad background mein chalayein
      (jaise LangMem ka `ReflectionExecutor` — user ko wait nahi karana).
- [ ] **Testing**: memory subsystem ko answer-quality se **alag** test karein — extraction
      accuracy, retrieval precision, aur isolation (cross-user leakage) sab check karein.
- [ ] **Deduplication/reconciliation**: contradictory facts ko handle karne ka plan rakhein
      (manual: purana fact dhoondo aur update karo; LangMem: `enable_deletes=True`).

---

# Aap Ne Kya Seekha (Recap)

1. **Short-term vs Long-term memory** ka farq — scope (thread vs user) ke terms mein.
2. Short-term memory ke 3 stages — full history (problem) -> sliding window -> summary+recent (production).
3. Long-term semantic memory — extraction + embeddings + similarity search + multi-user isolation.
4. Persistence — SQLite se restart-safe memory.
5. **RAG + Memory real integration** — query rewriting, retrieval, long-term facts, sab ek
   `RAGAssistant` class mein combined (Part 5 — apne RAG project mein isay directly use kar sakte hain).
6. LangGraph ka Checkpointer (thread-scoped) vs Store (user-scoped, cross-thread) — real graph ke sath.
7. LangMem — jab production scale par ready-made library use karni ho.

## Next Steps (Apne RAG Project Ke Liye)

1. Part 5 ki `RAGAssistant` class ko apne actual documents (PDF/Word/Notion) ke sath connect karein.
2. Agar aap ka app multi-user hai, `LongTermMemory` ki jagah `SQLiteLongTermMemory` ya Postgres
   use karein (restart-safe hona chahiye).
3. Agar aap ka system multi-step agent hai (sirf simple Q&A nahi), Part 6 ka LangGraph pattern
   apnayein — checkpointer + store dono sahi jagah use karein.
4. Jab team/scale badhe, Part 7 ka LangMem migrate karein taake reconciliation/deduplication
   automatically ho.
